# Weather Competition Jupyter Runner

Run these cells from top to bottom after uploading the project to JupyterLab. Start with the smoke test settings, then switch to full training after the environment and data paths are confirmed.

In [ ]:
# 1. Check runtime
import os
import sys
import torch

print(sys.version)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Check project and platform paths
from pathlib import Path

ROOT = Path('.').resolve()
print('ROOT:', ROOT)
print([p.name for p in ROOT.iterdir()])

!python scripts/check_platform_paths.py

In [ ]:
# 3. Set paths if the platform dataset is not under data/train and data/test
# os.environ['TRAIN_DIR'] = 'actual_train_dir'
# os.environ['TEST_DIR'] = 'actual_test_dir'
# os.environ['SAMPLE_SUBMISSION_PATH'] = 'actual_sample_submission.csv'

print('TRAIN_DIR =', os.environ.get('TRAIN_DIR', 'data/train'))
print('TEST_DIR =', os.environ.get('TEST_DIR', 'data/test'))
print('SAMPLE_SUBMISSION_PATH =', os.environ.get('SAMPLE_SUBMISSION_PATH', 'data/sample_submission.csv'))

In [ ]:
# 4. Check training data
from src.dataset import scan_image_folder

train_dir = os.environ.get('TRAIN_DIR', 'data/train')
df = scan_image_folder(train_dir)
print(df['label'].value_counts().sort_index())
print('total:', len(df))

In [ ]:
# 5. Lightweight smoke-test parameters for first JupyterLab run
os.environ['MODEL_NAME'] = 'efficientnet_b0'
os.environ['FALLBACK_MODEL_NAME'] = 'efficientnet_b0'
os.environ['PRETRAINED'] = 'false'
os.environ['IMG_SIZE'] = '224'
os.environ['BATCH_SIZE'] = '8'
os.environ['EPOCHS'] = '1'
os.environ['NUM_WORKERS'] = '0'

for key in ['MODEL_NAME', 'PRETRAINED', 'IMG_SIZE', 'BATCH_SIZE', 'EPOCHS', 'NUM_WORKERS']:
    print(key, os.environ[key])

In [ ]:
# 6. Run isolated synthetic smoke test
# This does not need official test images and writes generated files under tmp/smoke_test.
!python scripts/smoke_test.py --epochs 1 --batch-size 8 --model-name efficientnet_b0 --fallback-model-name efficientnet_b0

In [ ]:
# 7. Full-training configuration: start with rescue, then switch to high_score
# RUN_PROFILE='rescue' is the safest first run after a collapse; change to 'high_score' only after val_pred_distribution has all 4 classes.
if not torch.cuda.is_available():
    raise RuntimeError('Current Jupyter kernel cannot see CUDA. Restart Jupyter with the GPU PyTorch kernel first.')

RUN_PROFILE = 'rescue'  # options: 'rescue', 'high_score'

rescue_config = {
    'EXPERIMENT_NAME': 'rescue_e2s_224_light_ce_classweight',
    'DEVICE': 'cuda',
    'REQUIRE_CUDA': 'true',
    'MODEL_NAME': 'tf_efficientnetv2_s',
    'FALLBACK_MODEL_NAME': 'efficientnet_b0',
    'PRETRAINED': 'true',
    'IMG_SIZE': '224',
    'BATCH_SIZE': '16',
    'EPOCHS': '20',
    'LEARNING_RATE': '1e-4',
    'WEIGHT_DECAY': '1e-4',
    'LABEL_SMOOTHING': '0.03',
    'AUGMENT_PROFILE': 'light',
    'LOSS_TYPE': 'ce',
    'FOCAL_GAMMA': '2.0',
    'SAMPLER_TYPE': 'none',
    'USE_CLASS_WEIGHT': 'true',
    'USE_EMA': 'false',
    'EMA_DECAY': '0.999',
    'USE_WARMUP': 'true',
    'WARMUP_EPOCHS': '2',
    'USE_FREEZE_BACKBONE': 'false',
    'HEAD_LR_MULT': '1.0',
    'GRAD_ACCUM_STEPS': '1',
    'MAX_GRAD_NORM': '1.0',
    'USE_TTA': 'false',
    'TTA_MODE': 'none',
    'NUM_WORKERS': '0',
    'USE_AMP': 'true',
    'TARGET_METRIC': 'macro_f1',
}

high_score_config = {
    'EXPERIMENT_NAME': 'best_e2s_300_weather_focal_ema',
    'DEVICE': 'cuda',
    'REQUIRE_CUDA': 'true',
    'MODEL_NAME': 'tf_efficientnetv2_s',
    'FALLBACK_MODEL_NAME': 'efficientnet_b0',
    'PRETRAINED': 'true',
    'IMG_SIZE': '300',
    'BATCH_SIZE': '8',
    'EPOCHS': '40',
    'LEARNING_RATE': '2e-4',
    'WEIGHT_DECAY': '1e-4',
    'LABEL_SMOOTHING': '0.05',
    'AUGMENT_PROFILE': 'weather_safe',
    'LOSS_TYPE': 'focal',
    'FOCAL_GAMMA': '2.0',
    'SAMPLER_TYPE': 'none',
    'USE_CLASS_WEIGHT': 'false',
    'USE_EMA': 'true',
    'EMA_DECAY': '0.999',
    'USE_WARMUP': 'true',
    'WARMUP_EPOCHS': '3',
    'USE_FREEZE_BACKBONE': 'false',
    'HEAD_LR_MULT': '1.0',
    'GRAD_ACCUM_STEPS': '1',
    'MAX_GRAD_NORM': '1.0',
    'USE_TTA': 'false',
    'TTA_MODE': 'none',
    'NUM_WORKERS': '0',
    'USE_AMP': 'true',
    'TARGET_METRIC': 'macro_f1',
}

selected_config = rescue_config if RUN_PROFILE == 'rescue' else high_score_config
for key, value in selected_config.items():
    os.environ[key] = value

print('RUN_PROFILE =', RUN_PROFILE)
for key in selected_config:
    print(key, os.environ[key])

print('Do not combine SAMPLER_TYPE=weighted with USE_CLASS_WEIGHT=true or LOSS_TYPE=class_balanced_focal until the rescue run is stable.')
print('After every epoch, check val_pred_distribution. If it contains only cloudy, stop and keep the rescue profile.')



In [ ]:
# 8. Run full training in the current Jupyter kernel for live progress
# Set environment variables before importing/reloading config and train.
# Running train.main() in-kernel avoids subprocess buffering, so tqdm and epoch debug output appear live.
import importlib
import config
import train

importlib.reload(config)
importlib.reload(train)
train.main()



In [ ]:
# 9. Run inference after official test images are available
import sys
!"{sys.executable}" infer.py

In [ ]:
# 10. Download and prepare an external labeled public test set
# Source: Hugging Face davidshableski/weatherimages. The script keeps cloudy/rainy/snowy/sunny and skips sunrise.
import sys
!"{sys.executable}" scripts/download_public_weather_test.py --output-dir tmp/public_weather_test --max-per-class 50

In [ ]:
# 11. Evaluate the trained checkpoint on the external labeled test set
import sys
!"{sys.executable}" scripts/evaluate_labeled_folder.py --data-dir tmp/public_weather_test --output-dir outputs/external_test --batch-size 32 --num-workers 0
!"{sys.executable}" scripts/analyze_errors.py
!"{sys.executable}" scripts/benchmark_inference.py --batch-size 32

In [ ]:
# 12. Check generated files, submission CSV, and external test metrics
import pandas as pd

paths = [
    Path('results/best_model.pth'),
    Path('results/training_summary.json'),
    Path('logs/train_log.csv'),
    Path('outputs/submissions/submission.csv'),
    Path('outputs/external_test/external_test_metrics.json'),
    Path('outputs/confusion_matrix.png'),
]
for path in paths:
    print(path, path.exists())

sub_path = Path('outputs/submissions/submission.csv')
if sub_path.exists():
    sub = pd.read_csv(sub_path)
    print(sub.head())
    print(sub.shape)
    print(sub.isnull().sum())